## Script for Testing Segmentation CNN

# Imports

In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score, f1_score


from tensorflow.keras.layers import Conv1D, MaxPooling1D, Input, UpSampling1D, Concatenate
from tensorflow.keras.models import Sequential 

''' 
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
mne.set_log_level("CRITICAL")
''' 

' \ngpus = tf.config.list_physical_devices(\'GPU\')\nif gpus:\n    for gpu in gpus:\n        tf.config.experimental.set_memory_growth(gpu, True)\nmne.set_log_level("CRITICAL")\n'

In [ ]:
features_all = pd.read_pickle("training_features_18042026.pkl")
features_all = features_all[~((features_all["Subject"] == "RL03JG") & (features_all["Nap Number"] == 5))] # presentation for this subject is 0 ? somethign weird 

labels = pd.read_pickle("segments.pkl")

print("==========SANITY CHECK==========")
print("Subject rows align? ", (labels["Nap_ID"].values == features_all["Nap Number"].values).all())
print("Nap IDs align? ", labels["Subject"].values == features_all["Nap Subject"].values).all()


In [ ]:
# concatenate dfs
features_all = features_all.reset_index(drop=True)
labels = labels.reset_index(drop=True)

df_comb = pd.concat([features_all, labels], axis=1)  

## Functions

In [ ]:
def unet_1d(epoch_len, feature_num, num_classes):
    inputs = Input(shape=(epoch_len, feature_num))

    Model = Sequential([
        Input(shape=(epoch_len, feature_num)) # NEED TO MAKE SHAPE A PARAMETER 
         # binary
    ])

    # Encoder
    e1 = Conv1D(32, kernel_size=3, padding='same', activation='relu')(inputs)
    p1 = MaxPooling1D(pool_size=2)(e1)

    e2 = Conv1D(64, kernel_size=3, padding='same', activation='relu')(p1)
    p2 = MaxPooling1D(pool_size=2)(e2)

    # Bottleneck
    b = Conv1D(128, kernel_size=3, padding='same', activation='relu')(p2)

    # Decoder
    u1 = UpSampling1D(size=2)(b)
    u1 = Concatenate()([u1, e2])  # skip connection
    d1 = Conv1D(64, kernel_size=3, padding='same', activation='relu')(u1)

    u2 = UpSampling1D(size=2)(d1)
    u2 = Concatenate()([u2, e1])  # skip connection
    d2 = Conv1D(32, kernel_size=3, padding='same', activation='relu')(u2)

    # Output — one label per timestep
    outputs = Conv1D(num_classes, kernel_size=1, activation='softmax')(d2)

    return Model(inputs, outputs)

In [ ]:
def run_kfold_training(model_func,X_train_full, X, y,  input_shape, num_classes, feature_num,compile_kwargs,early_stop, fit_kwargs=None, # arguments fed to model.fit() 
                       n_splits=5,random_state=42, shuffle=True, verbose=1,epoch_num=10,
                         type=1,): # model type (1: single head count, 2: two head count and duration, 3: two head for zygo and corr, 4: four head) 
   # returns a dictionary with {"models", "histories", "cv_scores"}.     

    if fit_kwargs is None:
        fit_kwargs = {}
    fit_kwargs = fit_kwargs.copy()

 

    kf = KFold(n_splits=n_splits, random_state=random_state, shuffle=shuffle)
    models = []
    histories = []
    cv_scores = []

    fold = 1
    for train_idx, val_idx in kf.split(X_train_full):
        print(f"Fold: {fold} {'='*65}")
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

   
        model = model_func(input_shape, num_classes, feature_num)
        model.compile(**compile_kwargs)

        if type ==2:
            history = model.fit(
                    X_train,
                    {"count_output": y_train[:, 0],
                    "duration_output": y_train[:, 1],},
                        validation_data=(X_val,  {
                        "count_output": y_val[:, 0],
                        "duration_output": y_val[:, 1],
                    }
                    ),epochs=epoch_num,
                    verbose=verbose,
                )
            
            scores = model.evaluate(X_val, {
                                    "count_output": y_val[:, 0],
                                    "duration_output": y_val[:, 1],
                                    }, 
                                    return_dict=True)
            
        elif type == 3:
            history = model.fit(
                    X_train,
                    {"zygo_output": y_train[:, 0],
                    "corr_output": y_train[:, 1],},
                        validation_data=(X_val,  {
                        "zygo_output": y_val[:, 0],
                        "corr_output": y_val[:, 1],
                    }
                    ),epochs=epoch_num,
                    )
            
            scores = model.evaluate(X_val, {
            "zygo_output": y_val[:, 0],
            "corr_output": y_val[:, 1],
                },return_dict=True)
            
        elif type == 4:
              history = model.fit(X_train, {
                        'zygo_count_output': y_train[:, 0],
                        'zygo_duration_output': y_train[:, 1],
                        'corr_count_output': y_train[:, 2],
                        'corr_duration_output': y_train[:, 3]
                    }, epochs=epoch_num, validation_data=(
                    X_val,
                    {
                        'zygo_count_output': y_val[:, 0],
                        'zygo_duration_output': y_val[:, 1],
                        'corr_count_output': y_val[:, 2],
                        'corr_duration_output': y_val[:, 3]
                    }
                ), #callbacks=[early_stop])
                )
              
              scores = model.evaluate(X_val,{
                'zygo_count_output': y_val[:, 0],
                'zygo_duration_output': y_val[:, 1],
                'corr_count_output': y_val[:, 2],
                'corr_duration_output': y_val[:, 3]
            }, verbose=0, return_dict=True)            

        else:
            history = model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val), epochs=epoch_num,
            #callbacks=[early_stop],
        )
            
            scores = model.evaluate(X_val, y_val, return_dict=True)

        

        fold += 1
        models.append(model)
        histories.append(history)
        cv_scores.append(scores)

    return {
        "models": models,
        "histories": histories,
        "cv_scores": cv_scores,
    }

## Running Model

### Defining Training Data

In [ ]:
# when splitting epochs later 
muscle_groups = features_all["True_Muscle_Activated"].repeat(2).reset_index(drop=True)

X_zygo = features_all[["Zygo"]] 
X_corr = features_all[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=0)
y = np.concatenate([features_all["Segments_Zygo"].astype(int).to_numpy(),
                    features_all["Segments_Corr"].astype(int).to_numpy()])       

y_zygo = features_all["Segments_Zygo"].astype(int).to_numpy()
y_corr = features_all["Segments_Corr"].astype(int).to_numpy()

indices = np.arange(len(X))
idx_train, idx_test, muscle_train, muscle_test = train_test_split(indices, muscle_groups,test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X_zygo)[2]
epoch_len = np.shape(X_zygo)[1]

### Calling function for training 

In [ ]:
# call function for training
epoch_num = 10 

results_segmentation = run_kfold_training(
    model_func=unet_1d,
    X_train_full=X_train_full,
    X=X,
    y=y,
    input_shape=input_shape,
    num_classes=num_classes,
    feature_num=feature_num,
    compile_kwargs={
        "optimizer": "adam",
        "loss": "sparse_categorical_crossentropy",
        "metrics": ["accuracy"]
    },
    fit_kwargs={"epochs": 10},
    n_splits=5)

model_segmentation = results_segmentation["models"][-1]
model_historysegmentation = model_segmentation.fit(X_train_full, y_train_full, epochs=epoch_num, validation_data=(X_test, y_test)) #, callbacks=[early_stop])

## Results and Testing 

### K-means Results

In [ ]:
cvScores = model_segmentation["cv_scores"]
 

### Accuracy and F1

In [ ]:
corr_mask = muscle_test == "Corr"
zygo_mask = muscle_test == "Zygo"

corr_mask_tr = muscle_train == "Corr"
zygo_mask_tr = muscle_train == "Zygo"

# full train

[y_pred_train_contraction, y_pred_train_dur] = model_segmentation.predict(X_train_full)  
y_pred_train_contraction = np.argmax(y_pred_train_contraction, axis=1)   

# Predict on test data
[y_pred_test_contraction, y_pred_test_dur] = model_segmentation.predict(X_test)  
y_pred_test_contraction = np.argmax(y_pred_test_contraction, axis=1)   
 
# Calculate accuracy
accuracy_training = accuracy_score(y_train_full[:,0], y_pred_train_contraction)   
accuracy_tes = accuracy_score(y_test[:,0], y_pred_test_contraction)  

accuracy_training_zygo = accuracy_score(y_train_full[:,0][zygo_mask_tr], y_pred_train_contraction[zygo_mask_tr])   
accuracy_test_zygo = accuracy_score(y_test[:,0][zygo_mask], y_pred_test_contraction[zygo_mask])  

accuracy_training_corr = accuracy_score(y_train_full[:,0][corr_mask_tr], y_pred_train_contraction[corr_mask_tr])   
accuracy_test_corr = accuracy_score(y_test[:,0][corr_mask], y_pred_test_contraction[corr_mask] ) 

# Calculate F1 score
f1_training = f1_score(y_train_full[:,0], y_pred_train_contraction, average='weighted')  
f1_test = f1_score(y_test[:,0], y_pred_test_contraction, average='weighted')  

f1_training_zygo = f1_score(y_train_full[:,0][zygo_mask_tr], y_pred_train_contraction[zygo_mask_tr], average='weighted')   
f1_test_zygo = f1_score(y_test[:,0][zygo_mask], y_pred_test_contraction[zygo_mask], average='weighted')  

f1_training_corr = f1_score(y_train_full[:,0][corr_mask_tr], y_pred_train_contraction[corr_mask_tr], average='weighted')   
f1_test_corr = f1_score(y_test[:,0][corr_mask], y_pred_test_contraction[corr_mask], average='weighted') 


# Print accuracy and F1 score
print("Corru Scores------------------------------")  
print("Training Accuracy :", accuracy_training_corr) 
print("Test Accuracy :", accuracy_test_corr)  
print("Training F1 Score :", f1_training_corr)  
print("Test F1 Score :", f1_test_corr) 

print("Zygo Scores------------------------------")  
print("Training Accuracy :", accuracy_training_zygo)  
print("Test Accuracy :", accuracy_test_zygo)  
print("Training F1 Score :", f1_training_zygo)   
print("Test F1 Score :", f1_test_zygo)


### Plotting

In [ ]:
# one plot demonstrating labels against data 
# one plot showing calculated values against data 